# Batch data/output join

Collect raw batch JSONL payloads (directories named `p<digits>` under `labelling/artifacts/batches`) and batch output JSONL files (under `labelling/artifacts/batch_outputs`), then merge on `UT`/`custom_id` to inspect coverage and spot missing outputs.


In [1]:
import json
import re
from pathlib import Path

import pandas as pd

def find_repo_root(anchor="labelling"):
    """Walk upward until the anchor directory is found."""
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / anchor).exists():
            return parent
    raise FileNotFoundError(f"Could not find {anchor} directory above {here}")

repo_root = find_repo_root()
batches_root = repo_root / "labelling" / "artifacts" / "batches"
outputs_root = repo_root / "labelling" / "artifacts" / "batch_outputs"
batch_dir_pattern = re.compile(r"p\d+$")

print(f"Repo root: {repo_root}")
print(f"Batches root: {batches_root}")
print(f"Outputs root: {outputs_root}")


Repo root: /Users/hunain/d/coding-projects/biodiversity
Batches root: /Users/hunain/d/coding-projects/biodiversity/labelling/artifacts/batches
Outputs root: /Users/hunain/d/coding-projects/biodiversity/labelling/artifacts/batch_outputs


In [2]:
def load_jsonl(path: Path):
    """Yield parsed JSON objects from a JSONL file."""
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)

def extract_output_text(response_body):
    """Pull the first text block from response.body.output if present."""
    if not response_body or not isinstance(response_body, dict):
        return None

    for block in response_body.get("output") or []:
        for content in block.get("content") or []:
            text = content.get("text")
            if text is not None:
                return text
    return None


In [3]:
batch_dirs = sorted(
    [p for p in batches_root.iterdir() if p.is_dir() and batch_dir_pattern.match(p.name)],
    key=lambda p: int(p.name.lstrip("p")),
)

raw_records = []
raw_counts = []

for batch_dir in batch_dirs:
    data_dir = batch_dir / "data"
    rows_added = 0

    if data_dir.exists():
        for jsonl_file in sorted(data_dir.glob("*.jsonl")):
            for row in load_jsonl(jsonl_file):
                row["_batch"] = batch_dir.name
                row["_source_file"] = jsonl_file.name
                raw_records.append(row)
                rows_added += 1

    raw_counts.append({"batch": batch_dir.name, "raw_rows": rows_added})

raw_df = pd.DataFrame(raw_records)
raw_counts_df = pd.DataFrame(raw_counts).sort_values("batch")

print(f"Raw rows loaded: {len(raw_df):,}")
raw_counts_df


Raw rows loaded: 230,000


,batch,raw_rows
0,p1,10000
9,p10,10000
10,p11,10000
11,p12,10000
12,p13,10000
13,p14,10000
14,p15,10000
15,p16,10000
16,p17,10000
17,p18,10000


In [4]:
output_dirs = sorted(
    [p for p in outputs_root.iterdir() if p.is_dir() and batch_dir_pattern.match(p.name)],
    key=lambda p: int(p.name.lstrip("p")),
)

output_records = []
output_counts = []

for out_dir in output_dirs:
    rows_added = 0

    for jsonl_file in sorted(out_dir.glob("*.jsonl")):
        for row in load_jsonl(jsonl_file):
            response_body = (row.get("response") or {}).get("body")
            output_records.append({
                "custom_id": row.get("custom_id"),
                "output_text": extract_output_text(response_body),
                "_output_batch": out_dir.name,
                "_output_file": jsonl_file.name,
            })
            rows_added += 1

    output_counts.append({"batch": out_dir.name, "output_rows": rows_added})

output_df = pd.DataFrame(output_records)
output_counts_df = pd.DataFrame(output_counts).sort_values("batch")

if not output_df.empty:
    output_df = (
        output_df.sort_values(["_output_batch", "_output_file"])
        .drop_duplicates(subset=["custom_id"], keep="last")
        .reset_index(drop=True)
    )

print(f"Output rows loaded (de-duped by custom_id): {len(output_df):,}")
output_counts_df


Output rows loaded (de-duped by custom_id): 200,000


,batch,output_rows
0,p1,10000
9,p10,10000
10,p11,10000
11,p12,10000
12,p13,10000
13,p14,10000
14,p15,10000
15,p16,10000
16,p17,10000
17,p18,10000


In [5]:
if raw_df.empty:
    raise ValueError("No raw batch rows found; check paths and rerun.")

merged_df = raw_df.merge(
    output_df,
    how="left",
    left_on="UT",
    right_on="custom_id",
    suffixes=("", "_out"),
)

matched = merged_df["output_text"].notna().sum()
missing = merged_df["output_text"].isna().sum()

print(f"Rows after left join: {len(merged_df):,}")
print(f"Matched outputs: {matched:,}")
print(f"Missing outputs: {missing:,}")

coverage_by_batch = (
    merged_df.groupby("_batch")
    .agg(
        raw_rows=("UT", "size"),
        matched=("output_text", lambda s: s.notna().sum()),
    )
    .assign(missing=lambda df: df["raw_rows"] - df["matched"])
    .assign(coverage=lambda df: (df["matched"] / df["raw_rows"]).round(3))
    .sort_index()
)

coverage_by_batch


Rows after left join: 230,000
Matched outputs: 200,000
Missing outputs: 30,000


,raw_rows,matched,missing,coverage
_batch,,,,
p1,10000,10000,0,1.0
p10,10000,10000,0,1.0
p11,10000,10000,0,1.0
p12,10000,10000,0,1.0
p13,10000,10000,0,1.0
p14,10000,10000,0,1.0
p15,10000,10000,0,1.0
p16,10000,10000,0,1.0
p17,10000,10000,0,1.0


In [6]:
unmatched_outputs_df = (
    output_df[~output_df["custom_id"].isin(raw_df["UT"])]
    if not output_df.empty and not raw_df.empty
    else pd.DataFrame()
)

print(f"Outputs without a raw UT match: {len(unmatched_outputs_df):,}")

merged_df.head()


Outputs without a raw UT match: 0


,UT,title,authors,abstract,source,publication_year,wos_categories,doi,_batch,_source_file,custom_id,output_text,_output_batch,_output_file
0,WOS:001385022700001,Regional intermediary actors and professionali...,"[Hughes, P]",Community-led housing initiatives address hous...,"ROUTLEDGE JOURNALS, TAYLOR & FRANCIS LTD",2026,"[Environmental Studies, Regional & Urban Plann...",10.1080/02673037.2024.2439455,p1,batch-p1-001.jsonl,WOS:001385022700001,"{""results"": [""Not Applicable""]}",p1,batch_694a0e2ff4dc8190beb88034b82e160e_output....
1,WOS:001389849900001,The developmental changes in the otolith morph...,"[Koca, HU, Koca, SB, Jawad, LA]",Otoliths from four species of scorpaenids-Heli...,WILEY,2026,"[Anatomy & Morphology, Zoology]",10.1111/azo.12532,p1,batch-p1-001.jsonl,WOS:001389849900001,"{""results"": [""Not Applicable""]}",p1,batch_694a0e2ff4dc8190beb88034b82e160e_output....
2,WOS:001410940900001,Skeletal deformities in Sparus aurata from Tür...,"[Güclü, SS, Jawad, LA, Koca, HU, Cilbiz, M, Ay...",The first record of the abnormalities ankylosi...,WILEY,2026,"[Anatomy & Morphology, Zoology]",10.1111/azo.12538,p1,batch-p1-001.jsonl,WOS:001410940900001,"{""results"": [""Not Applicable""]}",p1,batch_694a0e2ff4dc8190beb88034b82e160e_output....
3,WOS:001416324100001,"Exploring diversity, abundance and ecological ...","[Alia, Z, Khechekhouche, E, Amara, DG, Messaou...",The following subject provides an in-depth loo...,WILEY,2026,"[Anatomy & Morphology, Zoology]",10.1111/azo.12539,p1,batch-p1-001.jsonl,WOS:001416324100001,"{""results"": [""Not Applicable""]}",p1,batch_694a0e2ff4dc8190beb88034b82e160e_output....
4,WOS:001427492700001,The relationships between fish length and otol...,"[Khan, S, Jawad, LA, Khan, MA, Park, JM, Bano,...",The relationship between the total length of f...,WILEY,2026,"[Anatomy & Morphology, Zoology]",10.1111/azo.12540,p1,batch-p1-001.jsonl,WOS:001427492700001,"{""results"": [""Not Applicable""]}",p1,batch_694a0e2ff4dc8190beb88034b82e160e_output....


In [18]:
import json
import ast
import pandas as pd
import re

def extract_results(x):
    if pd.isna(x):
        return []

    # already parsed?
    if isinstance(x, dict):
        obj = x
    else:
        s = str(x).strip()

        # (optional) remove ```json ... ``` fences if they exist
        s = re.sub(r'^```(?:json)?\s*|\s*```$', '', s, flags=re.IGNORECASE | re.DOTALL).strip()

        # try JSON first (works fine even with real newlines)
        try:
            obj = json.loads(s)
        except json.JSONDecodeError:
            # fallback: handles single quotes / python-ish dict strings
            try:
                obj = ast.literal_eval(s)
            except Exception:
                return []

    res = obj.get("results", [])
    if res is None:
        return []
    if not isinstance(res, list):
        res = [res]

    # strip extra whitespace/newlines from each item
    return [str(v).strip() for v in res]

merged_df["results_list"] = merged_df["output_text"].apply(extract_results)


In [15]:
merged_df["output_text_clean"] = (
    merged_df["output_text"]
      .astype("string")
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)


In [19]:
merged_df.head()

,UT,title,authors,abstract,source,publication_year,wos_categories,doi,_batch,_source_file,custom_id,output_text,_output_batch,_output_file,results_list,output_text_clean
0,WOS:001385022700001,Regional intermediary actors and professionali...,"[Hughes, P]",Community-led housing initiatives address hous...,"ROUTLEDGE JOURNALS, TAYLOR & FRANCIS LTD",2026,"[Environmental Studies, Regional & Urban Plann...",10.1080/02673037.2024.2439455,p1,batch-p1-001.jsonl,WOS:001385022700001,"{""results"": [""Not Applicable""]}",p1,batch_694a0e2ff4dc8190beb88034b82e160e_output....,[Not Applicable],"{""results"": [""Not Applicable""]}"
1,WOS:001389849900001,The developmental changes in the otolith morph...,"[Koca, HU, Koca, SB, Jawad, LA]",Otoliths from four species of scorpaenids-Heli...,WILEY,2026,"[Anatomy & Morphology, Zoology]",10.1111/azo.12532,p1,batch-p1-001.jsonl,WOS:001389849900001,"{""results"": [""Not Applicable""]}",p1,batch_694a0e2ff4dc8190beb88034b82e160e_output....,[Not Applicable],"{""results"": [""Not Applicable""]}"
2,WOS:001410940900001,Skeletal deformities in Sparus aurata from Tür...,"[Güclü, SS, Jawad, LA, Koca, HU, Cilbiz, M, Ay...",The first record of the abnormalities ankylosi...,WILEY,2026,"[Anatomy & Morphology, Zoology]",10.1111/azo.12538,p1,batch-p1-001.jsonl,WOS:001410940900001,"{""results"": [""Not Applicable""]}",p1,batch_694a0e2ff4dc8190beb88034b82e160e_output....,[Not Applicable],"{""results"": [""Not Applicable""]}"
3,WOS:001416324100001,"Exploring diversity, abundance and ecological ...","[Alia, Z, Khechekhouche, E, Amara, DG, Messaou...",The following subject provides an in-depth loo...,WILEY,2026,"[Anatomy & Morphology, Zoology]",10.1111/azo.12539,p1,batch-p1-001.jsonl,WOS:001416324100001,"{""results"": [""Not Applicable""]}",p1,batch_694a0e2ff4dc8190beb88034b82e160e_output....,[Not Applicable],"{""results"": [""Not Applicable""]}"
4,WOS:001427492700001,The relationships between fish length and otol...,"[Khan, S, Jawad, LA, Khan, MA, Park, JM, Bano,...",The relationship between the total length of f...,WILEY,2026,"[Anatomy & Morphology, Zoology]",10.1111/azo.12540,p1,batch-p1-001.jsonl,WOS:001427492700001,"{""results"": [""Not Applicable""]}",p1,batch_694a0e2ff4dc8190beb88034b82e160e_output....,[Not Applicable],"{""results"": [""Not Applicable""]}"


In [24]:
filtered = merged_df[merged_df["results_list"].apply(lambda x: "Not Applicable" in (x))]


In [25]:
len(merged_df)

230000

In [26]:
len(filtered)

107538

In [29]:
merged_df

,UT,title,authors,abstract,source,publication_year,wos_categories,doi,_batch,_source_file,custom_id,output_text,_output_batch,_output_file,results_list,output_text_clean
0,WOS:001385022700001,Regional intermediary actors and professionali...,"[Hughes, P]",Community-led housing initiatives address hous...,"ROUTLEDGE JOURNALS, TAYLOR & FRANCIS LTD",2026,"[Environmental Studies, Regional & Urban Plann...",10.1080/02673037.2024.2439455,p1,batch-p1-001.jsonl,WOS:001385022700001,"{""results"": [""Not Applicable""]}",p1,batch_694a0e2ff4dc8190beb88034b82e160e_output....,[Not Applicable],"{""results"": [""Not Applicable""]}"
1,WOS:001389849900001,The developmental changes in the otolith morph...,"[Koca, HU, Koca, SB, Jawad, LA]",Otoliths from four species of scorpaenids-Heli...,WILEY,2026,"[Anatomy & Morphology, Zoology]",10.1111/azo.12532,p1,batch-p1-001.jsonl,WOS:001389849900001,"{""results"": [""Not Applicable""]}",p1,batch_694a0e2ff4dc8190beb88034b82e160e_output....,[Not Applicable],"{""results"": [""Not Applicable""]}"
2,WOS:001410940900001,Skeletal deformities in Sparus aurata from Tür...,"[Güclü, SS, Jawad, LA, Koca, HU, Cilbiz, M, Ay...",The first record of the abnormalities ankylosi...,WILEY,2026,"[Anatomy & Morphology, Zoology]",10.1111/azo.12538,p1,batch-p1-001.jsonl,WOS:001410940900001,"{""results"": [""Not Applicable""]}",p1,batch_694a0e2ff4dc8190beb88034b82e160e_output....,[Not Applicable],"{""results"": [""Not Applicable""]}"
3,WOS:001416324100001,"Exploring diversity, abundance and ecological ...","[Alia, Z, Khechekhouche, E, Amara, DG, Messaou...",The following subject provides an in-depth loo...,WILEY,2026,"[Anatomy & Morphology, Zoology]",10.1111/azo.12539,p1,batch-p1-001.jsonl,WOS:001416324100001,"{""results"": [""Not Applicable""]}",p1,batch_694a0e2ff4dc8190beb88034b82e160e_output....,[Not Applicable],"{""results"": [""Not Applicable""]}"
4,WOS:001427492700001,The relationships between fish length and otol...,"[Khan, S, Jawad, LA, Khan, MA, Park, JM, Bano,...",The relationship between the total length of f...,WILEY,2026,"[Anatomy & Morphology, Zoology]",10.1111/azo.12540,p1,batch-p1-001.jsonl,WOS:001427492700001,"{""results"": [""Not Applicable""]}",p1,batch_694a0e2ff4dc8190beb88034b82e160e_output....,[Not Applicable],"{""results"": [""Not Applicable""]}"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229995,WOS:001155289100001,Increasing the seed production efficiency of a...,"[Singh, A, Aulakh, CS, Sidhu, AS]",Higher productivity of seed-size tubers is a s...,WILEY,2024,[Agronomy],10.1002/csc2.21194,p24,batch-p24-001.jsonl,NaN,NaN,NaN,NaN,[],<NA>
229996,WOS:001155290100001,"Biological activities, identification, method ...","[Naznin, M, Alam, R, Alam, MB, Jung, MJ, Lee, ...",Introduction: Nymphaea rubra belongs to the Ny...,WILEY,2024,"[Biochemical Research Methods, Plant Sciences,...",10.1002/pca.3329,p24,batch-p24-001.jsonl,NaN,NaN,NaN,NaN,[],<NA>
229997,WOS:001155290600001,Assessing air pollution changes during the COV...,"[Mokarram, M, Taripanah, F, Pham, TM]","The COVID-19 pandemic, spanning nearly two yea...",ELSEVIER SCI LTD,2024,"[Engineering, Aerospace, Astronomy & Astrophys...",10.1016/j.asr.2023.11.039,p24,batch-p24-001.jsonl,NaN,NaN,NaN,NaN,[],<NA>
229998,WOS:001155294300001,Assessing the impact of distributed snow water...,"[Beaton, AD, Han, M, Tolson, BA, Buttle, JM, M...",Accuracy of the Copernicus snow water equivale...,WILEY,2024,[Water Resources],10.1002/hyp.15075,p24,batch-p24-001.jsonl,NaN,NaN,NaN,NaN,[],<NA>
